In [ ]:
%pip install pyarrow

In [ ]:
# Importation des bibliothèques nécessaires
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# On charge le fichier de données
df = pd.read_parquet(r'C:\Users\manug\OneDrive\Desktop\ML_2026\train.parquet')

# Afficher les 5 premières lignes pour découvrir les noms exacts des colonnes
display(df.head())


In [18]:
# Remplace par ton vrai chemin absolu, comme tu l'as fait pour train.parquet
chemin_sensors = r"C:\Users\manug\OneDrive\Desktop\ML_2026\sensors.parquet"

df_sensors = pd.read_parquet(chemin_sensors)
display(df_sensors.head())

,sensor,coor_x,coor_y,coor_z
0,N2,0.5,0.0,0.0
1,N4,1.4,0.0,0.0
2,N5,0.5,2.4,0.0
3,N6,0.0,2.4,0.0
4,N7,0.0,3.5,0.0


In [19]:
# On fusionne les données d'entraînement avec les positions des capteurs
# 'on="sensor"' indique la colonne commune qui sert de clé
# 'how="left"' s'assure qu'on garde bien toutes nos lignes de temps
df_complet = pd.merge(df, df_sensors, on='sensor', how='left')

# On vérifie le résultat
display(df_complet.head())

,sensor,time,power,temperature,coor_x,coor_y,coor_z
0,N102,0.0,1487.964722,17.514429,46.131474,3.5,0.0
1,N102,864000.0,1487.288818,17.820795,46.131474,3.5,0.0
2,N102,1728000.0,1486.612915,17.573187,46.131474,3.5,0.0
3,N102,2592000.0,1485.936890,16.513235,46.131474,3.5,0.0
4,N102,3456000.0,1485.260986,16.303427,46.131474,3.5,0.0


In [ ]:
# Affiche la liste des capteurs et combien de fois ils apparaissent
df['sensor'].value_counts()

In [ ]:
# On isole uniquement les données du capteur N102
capteur_N102 = df[df['sensor'] == 'N102']

# On crée le graphique pour la puissance
plt.figure(figsize=(10, 5))
plt.plot(capteur_N102['time'], capteur_N102['power'], marker='.', linestyle='-')
plt.title('Évolution de la puissance pour le capteur N102')
plt.xlabel('Temps')
plt.ylabel('Puissance')
plt.grid(True)
plt.show()

In [ ]:
# On garde les données triées du capteur N102
capteur_N102 = df[df['sensor'] == 'N102'].sort_values(by='time')

# Création d'un graphique avec deux axes Y (pour avoir des échelles différentes)
fig, ax1 = plt.subplots(figsize=(12, 6))

color = 'tab:blue'
ax1.set_xlabel('Temps (secondes)')
ax1.set_ylabel('Puissance', color=color)
ax1.plot(capteur_N102['time'], capteur_N102['power'], color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # On crée un deuxième axe Y qui partage le même axe X
color = 'tab:red'
ax2.set_ylabel('Température (°C)', color=color)
ax2.plot(capteur_N102['time'], capteur_N102['temperature'], color=color, alpha=0.7)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Évolution de la puissance et de la température (Capteur N102)')
fig.tight_layout()
plt.show()

In [20]:
# Importation de l'outil scikit-learn pour séparer les données
from sklearn.model_selection import train_test_split

# 1. On isole la CIBLE (ce qu'on veut prédire : y)
y = df_complet['temperature']

# 2. On isole les CARACTÉRISTIQUES (les indices pour deviner : X)
# On garde le temps, la puissance, et les coordonnées 3D.
# On exclut le nom du capteur ("N102") car un algorithme mathématique ne sait lire que des nombres.
X = df_complet[['time', 'power', 'coor_x', 'coor_y', 'coor_z']]

# 3. On coupe le jeu de données en deux (80% pour apprendre, 20% pour l'examen)
# random_state=42 permet de faire une coupe aléatoire, mais qui sera la même à chaque fois que tu relances le code.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# On affiche le résultat pour vérifier
print(f"L'IA va s'entraîner sur {X_train.shape[0]} lignes.")
print(f"L'IA sera testée sur {X_test.shape[0]} lignes.")

L'IA va s'entraîner sur 5301542 lignes.
L'IA sera testée sur 1325386 lignes.


In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. On initialise notre nouveau modèle (plus intelligent)
# n_estimators=50 veut dire qu'on crée 50 arbres de décision. 
# n_jobs=-1 permet d'utiliser toute la puissance de ton processeur pour aller plus vite.
modele_rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)

# 2. L'entraînement
print("Entraînement de la Forêt Aléatoire en cours (patiente un peu, ça peut prendre 1 à 2 minutes)...")
modele_rf.fit(X_train, y_train)
print("Entraînement terminé ! 🌲🎉\n")

# 3. L'examen et la correction
predictions_rf = modele_rf.predict(X_test)
erreur_moyenne_rf = mean_absolute_error(y_test, predictions_rf)
score_r2_rf = r2_score(y_test, predictions_rf)

print("--- NOUVEAUX RÉSULTATS ---")
print(f"Erreur moyenne : L'IA se trompe de {erreur_moyenne_rf:.2f} °C en moyenne.")
print(f"Score R²       : {score_r2_rf:.2f}")

Entraînement de la Forêt Aléatoire en cours (patiente un peu, ça peut prendre 1 à 2 minutes)...
Entraînement terminé ! 🌲🎉

--- NOUVEAUX RÉSULTATS ---
Erreur moyenne : L'IA se trompe de 1.43 °C en moyenne.
Score R²       : 1.00


In [ ]:
# 1. Charger le fichier d'examen (test)
# N'oublie pas de mettre le vrai chemin de ton fichier
chemin_test = r"C:\Users\manug\OneDrive\Desktop\ML_2026\test.parquet" 
df_test = pd.read_parquet(chemin_test)

# 2. On fusionne avec les capteurs pour donner les coordonnées X, Y, Z à l'IA
df_test_complet = pd.merge(df_test, df_sensors, on='sensor', how='left')

# 3. On donne à l'IA exactement les mêmes colonnes qu'elle a apprises
X_examen = df_test_complet[['time', 'power', 'coor_x', 'coor_y', 'coor_z']]

# 4. L'IA passe l'examen et devine les températures !
predictions_finales = modele_rf.predict(X_examen)

# 5. On affiche les premières lignes de test pour voir à quoi elles ressemblent
display(df_test.head())

print("\nEt voici les 5 premières températures devinées par ton IA :")
print(predictions_finales[:5])